
# Generic scRNA‑seq Analysis Pipeline (10x Genomics, Scanpy)
This notebook provides a clean and reproducible template for performing a full single‑cell RNA‑seq analysis using Scanpy.

It is designed so that you can easily adapt it to any new 10x Genomics dataset by modifying only a few configuration fields at the top of the notebook.

(PS C:\cd K:\scRNA) (PS K:\scRNA> py -m notebook)

## Initial Setup (Required Before Running the Pipeline)
Before running any analysis, complete the following steps:
### 1. Create the project directory
Inside your working directory, in 'data' folder, create a new folder named after your project, for example:
#### K:\scRNA\data\YourProjectName\
Inside this folder, create the following subdirectories:
#### raw/ – place the original (compressed) raw data here.
#### data/YourProjectName/ – place the un‑zipped 10x output (e.g., filtered_feature_bc_matrix/) here.

### Project folder structure
scRNA/

├── raw/ raw_input_files or folders (.gz / .fastq / .tar.gz)

├── data/YYMMDD-new_project/matrix.mtx, features.tsv, barcodes.tsv
...

### 2. Provide essential project metadata
In the “Project Configuration” cell of this notebook, fill in the following fields:
project_name – a short unique identifier (e.g., "Heart_10x_2024").
tissue_name – the biological tissue or organ (e.g., "Heart", "PBMC", "Brain").
data_path – path to the unzipped 10x directory inside data/.
raw_path – path to the raw files inside raw/.
Only these parameters must be changed when analyzing a new dataset.

### 3. QC configuration
Define the quality‑control thresholds appropriate for your dataset, including:
minimum / maximum number of genes per cell
maximum mitochondrial gene percentage
minimum UMI counts
filtering for empty droplets or doublets (if required)
These thresholds appear in the “QC Settings” cell and can be adjusted dataset‑by‑dataset.

## Notebook1 Outline
1. Project configuration (template parameters)
2. Setup & imports + Load raw data and save initial adata
3. Compute QC metrics and visualize distributions
4. Filtering (cells & genes)
5. Normalization, log-transform, and `adata.raw`
6. Highly variable gene (HVG) selection
7. Scaling and PCA
8. Neighborhood graph construction
9. UMAP embedding
10. Leiden clustering (default + multiple resolutions)
11. Marker gene analysis and export
12. Cell type annotation (optional, dataset-specific)
13. Hierarchical annotation

-------
# Generic scRNA-seq Pipeline Template (10x Genomics, Scanpy)

This notebook is a **template** for a standard single-cell RNA-seq analysis using **Scanpy**.

You can adapt it to different 10x Genomics datasets by changing only a few parameters in the **Project configuration** cell:

#### - Make folders in the 'name of new project' in 'raw' and 'data' folders.--[Change(1) , Change(2)]
#### - InPut the 'raw-data' of new project in 'raw' folder and unziped data of new project in 'data' folder.
------
n1 = notebook 1 = Analysis ,
n2 = notebook 2 = Annotation ,
n3 = notebook 3 = GO ,

## notebook 1 - Analysis

In [ ]:
# ==========================================
# n1-1 — Project & Pipeline Configuration
# ==========================================

import yaml
from pathlib import Path

# ------------------------------------------
# a) Project metadata
# ------------------------------------------

# Name of your dataset folder (where results & figures go)
DATASET_NAME = "260429-pbmc3k"  # Input the date & name of project.

# Name of the config file (without .yaml)
CONFIG_NAME = "pbmc"   # or "tumor", "brain", "pancreas", ...

# ------------------------------------------
# b) Paths & directories
# ------------------------------------------

# p--  BASE_DIR = Path("../..").resolve()
BASE_DIR = Path.cwd().parents[1]

CONFIG_DIR = BASE_DIR / "configs"
RAW_DATA_DIR = BASE_DIR / "data" / "raw"
DATA_DIR = BASE_DIR / "data" / DATASET_NAME
RESULTS_DIR = BASE_DIR / "results" / DATASET_NAME
FIG_DIR = BASE_DIR / "figures" / DATASET_NAME

for p in [RESULTS_DIR, FIG_DIR]:
    p.mkdir(parents=True, exist_ok=True)

# ------------------------------------------
# c) Load configuration from YAML
# ------------------------------------------

config_path = CONFIG_DIR / f"{CONFIG_NAME}.yaml"

if not config_path.exists():
    raise FileNotFoundError(
        f"Config file not found: {config_path}\n"
        f"Make sure a YAML file exists in configs/ with this name."
    )

with open(config_path, "r") as f:
    CONFIG = yaml.safe_load(f)

# ------------------------------------------
# d) Extract configuration sections
# ------------------------------------------

META = CONFIG.get("meta", {})
QC = CONFIG.get("qc", {})
ANALYSIS = CONFIG.get("analysis", {})
ANNOTATION = CONFIG.get("annotation", {})
VIS = CONFIG.get("visualization", {})

# ------------------------------------------
# e) Global settings
# ------------------------------------------

RANDOM_SEED = CONFIG.get("random_seed", 0)
MULTI_RESOLUTIONS = CONFIG.get("multi_resolutions", [0.3, 0.5, 0.8, 1.0])
RANK_GENES_METHOD = CONFIG.get("rank_genes_method", "wilcoxon")
CLUSTER_KEY = CONFIG.get("cluster_key", "leiden")

# ------------------------------------------
# f) QC parameters
# ------------------------------------------

MIN_GENES_PER_CELL = QC.get("min_genes_per_cell")
MAX_GENES_PER_CELL = QC.get("max_genes_per_cell")
MAX_PCT_COUNTS_MT = QC.get("max_pct_mt")
MIN_CELLS_PER_GENE = QC.get("min_cells_per_gene")

# ------------------------------------------
# g) Analysis parameters
# ------------------------------------------

N_TOP_HVGS = ANALYSIS.get("n_top_hvgs")
N_PCS = ANALYSIS.get("n_pcs")
NEIGHBOR_K = ANALYSIS.get("neighbor_k")
DEFAULT_LEIDEN_RESOLUTION = ANALYSIS.get("leiden_resolution")

# ------------------------------------------
# h) Annotation & markers
# ------------------------------------------

REFERENCE_MARKERS = ANNOTATION.get("reference_markers", {})
MARKER_GENES_FOR_UMAP = VIS.get("umap_marker_genes", [])

# ------------------------------------------
# i) Summary printout
# ------------------------------------------

print("====================================")
print("   Configuration Loaded Successfully")
print("====================================")
print(f"Dataset name:       {DATASET_NAME}")
print(f"Loaded config file: {config_path.name}")
print()
print("QC:")
print(f"  min_genes_per_cell: {MIN_GENES_PER_CELL}")
print(f"  max_genes_per_cell: {MAX_GENES_PER_CELL}")
print(f"  max_pct_mt:         {MAX_PCT_COUNTS_MT}")
print()
print("Analysis:")
print(f"  HVGs:       {N_TOP_HVGS}")
print(f"  PCs:        {N_PCS}")
print(f"  neighbors:  {NEIGHBOR_K}")
print(f"  leiden res: {DEFAULT_LEIDEN_RESOLUTION}")
print()
print(f"Annotation marker groups: {len(REFERENCE_MARKERS)}")
print(f"UMAP marker genes:        {len(MARKER_GENES_FOR_UMAP)}")
print("====================================")


In [ ]:
# ==========================================
# n1-2 — Configuration Validation
# ==========================================

def validate_config():

    print("Running configuration validation...\n")

    # -------------------------
    # a) QC parameters
    # -------------------------

    required_qc = {
        "MIN_GENES_PER_CELL": MIN_GENES_PER_CELL,
        "MAX_GENES_PER_CELL": MAX_GENES_PER_CELL,
        "MAX_PCT_COUNTS_MT": MAX_PCT_COUNTS_MT,
        "MIN_CELLS_PER_GENE": MIN_CELLS_PER_GENE
    }

    for name, value in required_qc.items():
        if value is None:
            raise ValueError(f"Missing QC parameter: {name}")

    if MIN_GENES_PER_CELL >= MAX_GENES_PER_CELL:
        raise ValueError(
            "MIN_GENES_PER_CELL must be smaller than MAX_GENES_PER_CELL")


    if MAX_PCT_COUNTS_MT is not None:
        if not (0 <= MAX_PCT_COUNTS_MT <= 100):
          raise ValueError("MAX_PCT_COUNTS_MT must be between 0 and 100") 
        

    # -------------------------
    # b) Analysis parameters
    # -------------------------

    required_analysis = {
        "N_TOP_HVGS": N_TOP_HVGS,
        "N_PCS": N_PCS,
        "NEIGHBOR_K": NEIGHBOR_K,
        "DEFAULT_LEIDEN_RESOLUTION": DEFAULT_LEIDEN_RESOLUTION
    }

    for name, value in required_analysis.items():
        if value is None:
            raise ValueError(f"Missing analysis parameter: {name}")

    if N_TOP_HVGS <= 0:
        raise ValueError("N_TOP_HVGS must be > 0")

    if N_PCS <= 0:
        raise ValueError("N_PCS must be > 0")

    if NEIGHBOR_K <= 0:
        raise ValueError("NEIGHBOR_K must be > 0")

    # -------------------------
    # c) Marker genes
    # -------------------------

    if not isinstance(REFERENCE_MARKERS, dict):
        raise TypeError("REFERENCE_MARKERS must be a dictionary")

    for celltype, genes in REFERENCE_MARKERS.items():

        if not isinstance(genes, list):
            raise TypeError(
                f"Markers for {celltype} must be a list of genes"
            )

        if len(genes) == 0:
            raise ValueError(
                f"Marker list for {celltype} is empty"
            )

    # -------------------------
    # d) UMAP markers
    # -------------------------

    if not isinstance(MARKER_GENES_FOR_UMAP, list):
        raise TypeError("MARKER_GENES_FOR_UMAP must be a list")

    # -------------------------
    # e) Directory check
    # -------------------------

    if not DATA_DIR.exists():
        print(f"Warning: dataset directory does not exist yet: {DATA_DIR}")

    print("Configuration validation passed.\n")


validate_config()


In [ ]:
# -------------------------------------------------------------
# n1-2-1. Setup, imports, paths, and reproducibility
# -------------------------------------------------------------

import os
import importlib.metadata
from pathlib import Path

import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns
import ipywidgets
import gseapy as gp

# ------------------------------------------------------------------
# a) Scanpy settings
# ------------------------------------------------------------------
sc.settings.verbosity = 3          # 0: errors, 1: warnings, 2: info, 3: hints
sc.settings.n_jobs = 1
sc.set_figure_params(
    dpi=300,
    frameon=False,
    facecolor="white",
    format="png"
)

# ------------------------------------------------------------------
# b) Displaying Paths and Configuration Summary
# ------------------------------------------------------------------
print("Current working directory:", Path.cwd())
print("Base directory:", BASE_DIR)
print("Raw data directory:", RAW_DATA_DIR)
print("Data directory:", DATA_DIR)
print("Results directory:", RESULTS_DIR)
print("Figures directory:", FIG_DIR)

print(f"\nDataset: {DATASET_NAME}")
print(f"Main clustering key: {CLUSTER_KEY}")
print(f"Default Leiden resolution: {DEFAULT_LEIDEN_RESOLUTION}")

# ------------------------------------------------------------------
# c) Software versions (for reproducibility)
# ------------------------------------------------------------------
print("\nPackage versions:")
for pkg in ["scanpy", "anndata", "pandas", "numpy", "matplotlib", "seaborn", "ipywidgets", "gseapy"]:
    try:
        ver = importlib.metadata.version(pkg)
        print(f"  - {pkg}: {ver}")
    except importlib.metadata.PackageNotFoundError:
        print(f"  - {pkg}: NOT INSTALLED")


np.random.seed(RANDOM_SEED)
sc.settings.seed = RANDOM_SEED  



# Save enviroment

env_path = RESULTS_DIR / "0-environment.txt"
with open(env_path, "w") as f:
    for pkg in ["scanpy", "anndata", "pandas", "numpy", "matplotlib", "seaborn", "ipywidgets", "gseapy"]:
        try:
            ver = importlib.metadata.version(pkg)
            f.write(f"{pkg}=={ver}\n")
        except:
            pass


In [ ]:
# ------------------------------------------------------------------
# n1-2-2- Pretty printing of configuration (Rich formatting) (Optional)
# ------------------------------------------------------------------

from rich.console import Console
from rich.panel import Panel
from rich.table import Table
from rich.text import Text

console = Console()

# -------------------------------------------------
# a) Project / Dataset info panel
# -------------------------------------------------
header = Text()
header.append("scRNA-seq Pipeline\n", style="bold cyan")
header.append(f"Dataset: {DATASET_NAME}\n", style="bold white")
header.append(f"Clustering key: {CLUSTER_KEY}\n", style="white")
header.append(f"Default Leiden resolution: {DEFAULT_LEIDEN_RESOLUTION}", style="white")

console.print(
    Panel(
        header,
        title="Project Summary",
        border_style="cyan",
        expand=False
    )
)

# -------------------------------------------------
# b) Paths table
# -------------------------------------------------
paths_table = Table(title="Project Paths", show_header=True, header_style="bold magenta")

paths_table.add_column("Name", style="bold")
paths_table.add_column("Path", style="green")

paths_table.add_row("Base directory", str(BASE_DIR))
paths_table.add_row("Raw data", str(RAW_DATA_DIR))
paths_table.add_row("Processed data", str(DATA_DIR))
paths_table.add_row("Results", str(RESULTS_DIR))
paths_table.add_row("Figures", str(FIG_DIR))

console.print(paths_table)

# -------------------------------------------------
# c) Analysis parameters table
# -------------------------------------------------
params_table = Table(title="Key Analysis Parameters", show_header=True, header_style="bold yellow")

params_table.add_column("Parameter", style="bold")
params_table.add_column("Value", style="white")

params_table.add_row("Clustering method", "Leiden")
params_table.add_row("Cluster key", CLUSTER_KEY)
params_table.add_row("Default resolution", str(DEFAULT_LEIDEN_RESOLUTION))
params_table.add_row("Random seed", "Defined in config cell")

console.print(params_table)


### n1-2‑3. Load raw 10x data

In this step we load the raw gene expression matrix generated by **10x Genomics** using `scanpy.read_10x_mtx`.

The function reads the matrix files located in `DATA_DIR` and creates an **AnnData object (`adata`)** that will be used throughout the analysis.

Key details:

- `var_names="gene_symbols"`  
  Use gene symbols as feature names instead of Ensembl IDs.

- `adata.var_names_make_unique()`  
  Ensures that duplicated gene symbols are made unique, which is required for downstream analysis.

Make sure that the directory specified by `DATA_DIR` contains the standard **10x matrix files**, typically:
 matrix.mtx, barcodes.tsv, features.tsv

Older datasets may contain `genes.tsv` instead of `features.tsv`.

Example structure:
data/

└── YYMMDD-new_project/matrix.mtx, barcodes.tsv, features.tsv


After loading, the object `adata` will contain:

- cells in `adata.obs`
- genes in `adata.var`
- count matrix in `adata.X`


In [ ]:
# n1-2-3. Load raw 10x dataset

adata = sc.read_10x_mtx(
    DATA_DIR,
    var_names="gene_symbols",
    cache=True
)
adata.var_names_make_unique()

adata.uns["dataset"] = DATASET_NAME

print(adata)


adata.uns["pipeline"] = {
    "dataset": DATASET_NAME,
    "config": CONFIG_NAME,
    "date": pd.Timestamp.now().isoformat(),
    "random_seed": RANDOM_SEED
}


# Save raw checkpoint (We save the raw AnnData object so that we can always go back to the original data without re-reading the 10x files.)

raw_h5ad_path = RESULTS_DIR / f"1-{DATASET_NAME}_raw.h5ad"
adata.write(raw_h5ad_path)
print(f"Raw AnnData saved to: {raw_h5ad_path}")


In [ ]:
# # n1-2-4. Detection of organism

def detect_organism_from_adata(adata, n_check=50):
    genes = list(adata.var_names[:n_check]) + list(adata.var_names[-n_check:])

    if any(g.startswith("ENSG") for g in genes):
        return "human"
    if any(g.startswith("ENSMUSG") for g in genes):
        return "mouse"

    human_like = sum(g.isupper() for g in genes)
    mouse_like = sum(g[:1].isupper() and g[1:].islower() for g in genes)

    if human_like > mouse_like:
        return "human"
    elif mouse_like > human_like:
        return "mouse"

    return "unknown"


def normalize_organism(org):
    org = org.lower().strip()

    mapping = {
        "homo sapiens": "human",
        "hsapiens": "human",
        "human": "human",
        "mus musculus": "mouse",
        "mmusculus": "mouse",
        "mouse": "mouse"
    }

    return mapping.get(org, org)


# --- detect ---
ORGANISM_DEFAULT = detect_organism_from_adata(adata)

if ORGANISM_DEFAULT == "unknown":
    print("[WARNING] Could not confidently detect organism.")
    print("[WARNING] Falling back to 'human' (please verify).")
    ORGANISM_DEFAULT = "human"

# --- normalize ---
ORGANISM_DEFAULT = normalize_organism(ORGANISM_DEFAULT)

# --- store ---
adata.uns["organism"] = ORGANISM_DEFAULT

print(f"[INFO] Organism set to: {ORGANISM_DEFAULT}")

### n1-3-1. Quality control (QC): compute metrics

We compute standard QC metrics per cell:

- `n_genes_by_counts`: number of detected genes
- `total_counts`: total UMIs per cell
- `pct_counts_mt`: fraction of counts in mitochondrial genes


In [ ]:
# n1-3-1. Quality control (QC): compute metrics

# Identify mitochondrial genes (for human: prefix "MT-")
# p-- adata.var["mt"] = adata.var_names.str.startswith("MT-")
adata.var["mt"] = adata.var_names.str.upper().str.startswith("MT-")

# Compute QC metrics
sc.pp.calculate_qc_metrics(
    adata,
    qc_vars=["mt"],
    percent_top=None,
    log1p=False,
    inplace=True
)

adata.obs[["n_genes_by_counts", "total_counts", "pct_counts_mt"]].head()


# Optional: Doublet detection (Scrublet)
# Skipped in this version for simplicity
# Recommended for large datasets or publication


## n1-3-2. QC summary statistics

We inspect basic summary statistics of the QC metrics to get an idea of the data quality distribution.


In [ ]:
# n1-3-2. QC summary statistics

qc_columns = ["n_genes_by_counts", "total_counts", "pct_counts_mt"]
qc_summary = adata.obs[qc_columns].describe()
qc_summary


## n1-3-3. QC visualization

We visualize QC metrics to choose appropriate filtering thresholds:

- Violin plots
- Scatter plots

Thresholds are defined in the configuration cell:

- `MIN_GENES_PER_CELL`
- `MAX_GENES_PER_CELL`
- `MAX_PCT_COUNTS_MT`


In [ ]:
# n1-3-3. QC visualization

# Violin plots
sc.pl.violin(
    adata,
    ["n_genes_by_counts", "total_counts", "pct_counts_mt"],
    jitter=0.4,
    multi_panel=True,
    log=True,
    show=False
)
qc_violin_path = FIG_DIR / f"1-{DATASET_NAME}_QC_violin.png"
plt.savefig(qc_violin_path, bbox_inches="tight")
plt.close()
print(f"QC violin plot saved to: {qc_violin_path}")

# Scatter: total_counts vs pct_counts_mt
sc.pl.scatter(
    adata,
    x="total_counts",
    y="pct_counts_mt",
    show=False
)
qc_scatter_mt_path = FIG_DIR / f"2-{DATASET_NAME}_QC_pct_counts_mt.png"
plt.savefig(qc_scatter_mt_path, bbox_inches="tight")
plt.close()
print(f"QC scatter (pct_counts_mt) saved to: {qc_scatter_mt_path}")

# Scatter: total_counts vs n_genes_by_counts
sc.pl.scatter(
    adata,
    x="total_counts",
    y="n_genes_by_counts",
    show=False
)
qc_scatter_genes_path = FIG_DIR / f"3-{DATASET_NAME}_QC_n_genes_by_counts.png"
plt.savefig(qc_scatter_genes_path, bbox_inches="tight")
plt.close()
print(f"QC scatter (n_genes_by_counts) saved to: {qc_scatter_genes_path}")


## n1-4) Filter low-quality cells and genes

We apply QC filters defined in the configuration cell:

- Remove cells with:
  - fewer than `MIN_GENES_PER_CELL` genes
  - more than `MAX_GENES_PER_CELL` genes
  - mitochondrial percentage above `MAX_PCT_COUNTS_MT`
- Remove genes expressed in fewer than `MIN_CELLS_PER_GENE` cells.

You should adapt these thresholds to your dataset and QC plots.


In [ ]:
# n1-4) Filter low-quality cells and genes

# Filter cells: minimum genes
sc.pp.filter_cells(adata, min_genes=MIN_GENES_PER_CELL)

# Filter cells: maximum genes (potential doublets)
if MAX_GENES_PER_CELL is not None:
    adata = adata[adata.obs["n_genes_by_counts"] < MAX_GENES_PER_CELL, :].copy()

# Filter cells by mitochondrial content
if MAX_PCT_COUNTS_MT is not None:
    adata = adata[adata.obs["pct_counts_mt"] < MAX_PCT_COUNTS_MT, :].copy()

# Filter genes: minimum cells
sc.pp.filter_genes(adata, min_cells=MIN_CELLS_PER_GENE)

print(adata)


#  Save QC-filtered AnnData object as a checkpoint

qc_filtered_path = RESULTS_DIR / f"2-{DATASET_NAME}_qc_filtered.h5ad"
adata.write(qc_filtered_path)
print(f"QC-filtered AnnData saved to: {qc_filtered_path}")

## n1-5. Normalization and log-transform and Save

We perform:

1. Library-size normalization (`sc.pp.normalize_total`).
2. Log1p transform (`log(x + 1)`).
3. Store the normalized, log-transformed data in `adata.raw` for later use.


In [ ]:
# n1-5. Normalization and log-transform and Save

# Normalize total counts per cell
# p-- sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.normalize_total(adata, target_sum=1e4, inplace=True)

# Log1p transform
sc.pp.log1p(adata)

# Store raw normalized data
adata.raw = adata

adata.layers["counts"] = adata.X.copy()


norm_log_h5ad_path = RESULTS_DIR / f"3-{DATASET_NAME}_norm_log.h5ad"
adata.write(norm_log_h5ad_path)
print(f"Normalized + log-transformed AnnData saved to: {norm_log_h5ad_path}")


## n1-6) Highly variable genes (HVGs)

We identify highly variable genes (HVGs) using:

- `flavor="cell_ranger"`
- `n_top_genes = N_TOP_HVGS` (defined in the configuration cell)

Then we subset the data to HVGs only.


In [ ]:
# n1-6) Highly variable genes (HVGs)

# Identify highly variable genes
sc.pp.highly_variable_genes(
    adata,
# p--    flavor="cell_ranger",
    flavor="seurat_v3",
    n_top_genes=N_TOP_HVGS
)

print(f"Number of HVGs: {adata.var['highly_variable'].sum()}")

# Plot HVGs
sc.pl.highly_variable_genes(adata, show=False)
hvg_fig_path = FIG_DIR / f"4-{DATASET_NAME}_highly_variable_genes.png"
plt.savefig(hvg_fig_path, bbox_inches="tight")
plt.close()
print(f"HVG plot saved to: {hvg_fig_path}")

# Subset to HVGs
adata = adata[:, adata.var["highly_variable"]].copy()
print(adata)


## n1-7) Scaling and PCA

We:

- Scale each gene to unit variance and zero mean.
- Run PCA (with `N_PCS` components).
- Inspect the explained variance ratio.


In [ ]:
# n1-7) Scaling and PCA

# Scale the data
#sc.pp.regress_out(adata, ["total_counts", "pct_counts_mt"])
sc.pp.scale(adata, max_value=10)

# Run PCA
sc.tl.pca(adata, svd_solver="arpack", n_comps=N_PCS)


# PCA plot (variance ratio)
sc.pl.pca_variance_ratio(
    adata,
    log=True,
    n_pcs=N_PCS,
    show=False
)
pca_var_fig_path = FIG_DIR / f"5-{DATASET_NAME}_pca_variance_ratio.png"
plt.savefig(pca_var_fig_path, bbox_inches="tight")
plt.close()
print(f"PCA variance ratio plot saved to: {pca_var_fig_path}")


## n1-8) Neighborhood graph and save

We compute the k-nearest neighbors graph in PCA space using `N_PCS` PCs.


In [ ]:
# n1-8) Neighborhood graph and save

sc.pp.neighbors(
    adata,
    n_neighbors=NEIGHBOR_K,
    n_pcs=N_PCS
)

pca_neighbors_h5ad_path = RESULTS_DIR / f"4-{DATASET_NAME}_pca_neighbors.h5ad"
adata.write(pca_neighbors_h5ad_path)
print(f"PCA + neighbors checkpoint saved to: {pca_neighbors_h5ad_path}")


## n1-9) UMAP embedding

We compute a 2D UMAP embedding for visualization and save a figure.


In [ ]:
# n1-9) UMAP embedding

sc.tl.umap(adata, random_state=RANDOM_SEED)
# p-- sc.tl.umap(adata)

sc.pl.umap(
    adata,
    color=["n_genes_by_counts", "pct_counts_mt"],
    frameon=False,
    show=False
)
umap_fig_path = FIG_DIR / f"6-{DATASET_NAME}_umap_qc.png"
plt.savefig(umap_fig_path, bbox_inches="tight")
plt.close()
print(f"UMAP plot saved to: {umap_fig_path}")


# n1-10-1) Leiden clustering (default resolution) and UMAP

We perform Leiden clustering on the neighborhood graph:

- Default resolution: `DEFAULT_LEIDEN_RESOLUTION`.
- Cluster labels are stored in `adata.obs[CLUSTER_KEY]`.
- We visualize cluster assignments on the UMAP.


In [ ]:
# n1-10-1) Leiden clustering (default resolution) and UMAP

# (اختیاری) UMAP colored by selected marker genes
if len(MARKER_GENES_FOR_UMAP) > 0:
    genes_in_data = [g for g in MARKER_GENES_FOR_UMAP if g in adata.var_names]
    missing_genes = [g for g in MARKER_GENES_FOR_UMAP if g not in adata.var_names]

    if len(genes_in_data) > 0:
        print(f"Plotting UMAP for {len(genes_in_data)} marker genes.")
        if missing_genes:
            print(f"These marker genes were not found in var_names and will be skipped: {missing_genes}")

        sc.pl.umap(
            adata,
            color=genes_in_data,
            frameon=False,
            show=False
        )
        marker_umap_path = FIG_DIR / f"8-{DATASET_NAME}_UMAP_marker_genes.png"
        plt.savefig(marker_umap_path, bbox_inches="tight")
        plt.close()
        print(f"UMAP marker gene plot saved to: {marker_umap_path}")
    else:
        print("None of MARKER_GENES_FOR_UMAP were found in var_names. Skipping marker UMAP.")
# (اختیاری)


sc.tl.leiden(
    adata,
    resolution=DEFAULT_LEIDEN_RESOLUTION,
    key_added=CLUSTER_KEY,
    flavor="igraph",
# p--  n_iterations=2,
    n_iterations=-1,
    directed=False
)

print(f"Leiden clustering completed with resolution={DEFAULT_LEIDEN_RESOLUTION}.")
print(f"Cluster labels stored in adata.obs['{CLUSTER_KEY}'].")

sc.pl.umap(
    adata,
    color=[CLUSTER_KEY],
    legend_loc="on data",
    frameon=False,
    show=False
)
leiden_umap_path = FIG_DIR / f"7-{DATASET_NAME}_Leiden_clustering_UMAP.png"
plt.savefig(leiden_umap_path, bbox_inches="tight")
plt.close()
print(f"UMAP with Leiden clusters saved to: {leiden_umap_path}")


## n1-10-2) Leiden clustering at multiple resolutions (optional)

We run Leiden at several resolutions for comparison and visualize them on UMAP.

Resolutions are defined in `MULTI_RESOLUTIONS`.


In [ ]:
# n1-10-2) Leiden clustering at multiple resolutions (optional)

for r in MULTI_RESOLUTIONS:
    key = f"leiden_{r}"
    sc.tl.leiden(
        adata,
        resolution=r,
        key_added=key,
        flavor="igraph",
# p--   n_iterations=2,
        n_iterations=-1,
        directed=False
    )
    print(f"Leiden clustering completed for resolution={r}, stored in adata.obs['{key}'].")

sc.pl.umap(
    adata,
    color=[f"leiden_{r}" for r in MULTI_RESOLUTIONS],
    frameon=False,
    show=False
)

sc.tl.dendrogram(adata, groupby=CLUSTER_KEY)


multi_res_umap_path = FIG_DIR / f"8-{DATASET_NAME}_Leiden_multi_resolutions_UMAP.png"
plt.savefig(multi_res_umap_path, bbox_inches="tight")
plt.close()
print(f"UMAP with multiple Leiden resolutions saved to: {multi_res_umap_path}")


## n1-11-1) Marker gene analysis (per cluster)

We identify marker genes for each cluster defined by `CLUSTER_KEY` using the method specified in `RANK_GENES_METHOD` (e.g. `"wilcoxon"`).

We also export all marker genes to a CSV file.


In [ ]:
# n1-11) Marker gene analysis (per cluster)

# Rank genes per cluster
sc.tl.rank_genes_groups(
    adata,
    groupby=CLUSTER_KEY,
    method=RANK_GENES_METHOD,
    pts=True
)
print(f"Marker gene analysis completed ({RANK_GENES_METHOD}, grouped by '{CLUSTER_KEY}').")

# Plot top marker genes
sc.pl.rank_genes_groups(
    adata,
    n_genes=20,
    sharey=False,
    show=False
)
marker_fig_path = FIG_DIR / f"9-{DATASET_NAME}_Top_marker_genes.png"
plt.savefig(marker_fig_path, bbox_inches="tight")
plt.close()
print(f"Top marker genes plot saved to: {marker_fig_path}")

# Export markers to CSV
markers_df = sc.get.rank_genes_groups_df(
    adata,
    group=None  # all clusters
)
markers_csv_path = RESULTS_DIR / f"5-{DATASET_NAME}_marker_genes.csv"
markers_df.to_csv(markers_csv_path, index=False)


print(f"Marker genes table exported to CSV: {markers_csv_path}")
markers_df.head()


#  Save the processed AnnData object (with normalization, HVGs, PCA, neighbors, UMAP, Leiden, and marker statistics)
#   as a checkpoint. (before annotation)

processed_h5ad_path = RESULTS_DIR / f"6-{DATASET_NAME}_adata_processed.h5ad"
adata.write(processed_h5ad_path)
print(f"Processed AnnData saved to: {processed_h5ad_path}")


In [ ]:
## END of notebook 1- Analysis

In [ ]:
## ---------------------------------------

In [ ]:
## notebook 2 - Annotation 

In [ ]:
# n2-12) Automatic cell-type annotation using REFERENCE_MARKERS + score_genes

print("\n[Step 12] Automatic cell-type annotation using REFERENCE_MARKERS and sc.tl.score_genes")

# اگر داده خام در adata.raw ذخیره شده باشد، برای scoring از آن استفاده می‌کنیم
if adata.raw is not None:
    expr_adata = adata.raw.to_adata()
    print("Using adata.raw for marker scoring.")
else:
    expr_adata = adata
    print("Using adata (no .raw) for marker scoring.")

# بررسی اولیه: آیا REFERENCE_MARKERS پر شده است؟
if not REFERENCE_MARKERS:
    print("WARNING: REFERENCE_MARKERS is empty. "
          "Automatic annotation will be skipped. "
          "Check DATASET_TYPE and marker configuration.")
else:
    import scanpy as sc
    import numpy as np

    available_genes = set(expr_adata.var_names)
    print(f"Number of genes available for scoring: {len(available_genes)}")

    # a. محاسبه score برای هر cell type بر اساس REFERENCE_MARKERS
    celltypes = []
    marker_lists = []
    filtered_marker_lists = []

    for ct, genes in REFERENCE_MARKERS.items():
        genes = list(set(genes))  # unique
        present = [g for g in genes if g in available_genes]
        missing = [g for g in genes if g not in available_genes]

        if len(present) == 0:
            print(f"  [WARN] No marker genes found in data for cell type '{ct}'. "
                  f"Original markers: {genes}")
            continue

        print(f"  Cell type '{ct}': {len(present)} markers present, {len(missing)} missing.")
        if missing:
            print(f"    Missing: {missing}")

        celltypes.append(ct)
        marker_lists.append(genes)
        filtered_marker_lists.append(present)

        # محاسبه score این cell type برای تک تک سلول‌ها
        score_key = f"score_{ct}"
        sc.tl.score_genes(
            expr_adata,
            gene_list=present,
            score_name=score_key,
            use_raw=False  # چون expr_adata را از قبل تعیین کرده‌ایم
        )

        # انتقال score به adata.obs (برای اینکه بعداً با adata کار کنیم)
        adata.obs[score_key] = expr_adata.obs[score_key].copy()

    if len(celltypes) == 0:
        print("No valid marker sets found in REFERENCE_MARKERS. "
              "Skipping automatic cell-type assignment.")
    else:
        # b. انتخاب cell type غالب برای هر سلول
        score_cols = [f"score_{ct}" for ct in celltypes]
        score_matrix = adata.obs[score_cols].to_numpy()

        # argmax روی محور cell-type
        score_matrix = (score_matrix - score_matrix.mean(axis=1, keepdims=True)) / \
               (score_matrix.std(axis=1, keepdims=True) + 1e-9)
        
        best_idx = np.argmax(score_matrix, axis=1)
        best_scores = score_matrix[np.arange(score_matrix.shape[0]), best_idx]
        best_celltypes = np.array(celltypes)[best_idx]

        # می‌توان یک آستانه بگذاریم تا سلول‌هایی با score خیلی پایین "Unknown" شوند
        # این آستانه را می‌توانید تنظیم کنید (مثلاً 0.1، 0 یا None برای بدون فیلتر)
        SCORE_THRESHOLD = 0.25

        final_celltypes = []
        for s, ct in zip(best_scores, best_celltypes):
            if s < SCORE_THRESHOLD:
                final_celltypes.append("Unknown")
            else:
                final_celltypes.append(ct)

        adata.obs["cell_type"] = final_celltypes
        print("Cell-type assignment completed. Result stored in adata.obs['cell_type'].")

        # --------------------------------------------------
        # f) Cluster-level annotation (majority voting)
        # --------------------------------------------------
        def safe_majority(x):
            vc = x.value_counts()
            return vc.idxmax() if len(vc) > 0 else "Unknown"
        
        cluster_majority = (
            adata.obs.groupby(CLUSTER_KEY, observed=True)["cell_type"]
            .agg(safe_majority)
        )

        adata.obs["cell_type_cluster"] = adata.obs[CLUSTER_KEY].map(cluster_majority)
        
        print("Cluster-level annotation added: adata.obs['cell_type_cluster']")


        # c. خلاصه تعداد سلول‌ها در هر cell type
        print("\nCell counts per assigned cell type:")
        print(adata.obs["cell_type"].value_counts())

        # d. رسم UMAP با رنگ سل‌تایپ
        sc.pl.umap(adata, color=["cell_type"], frameon=False, show=False)
        celltype_umap_path = FIG_DIR / f"10-{DATASET_NAME}_UMAP_cell_types.png"
        plt.savefig(celltype_umap_path, bbox_inches="tight")
        plt.close()
        print(f"UMAP colored by cell type saved to: {celltype_umap_path}")

       
        # e. ذخیره AnnData شامل cell_type
        annotated_h5ad_path = RESULTS_DIR / f"7-{DATASET_NAME}_adata_annotated.h5ad"
        adata.write(annotated_h5ad_path)
        print(f"Annotated AnnData with cell_type saved to: {annotated_h5ad_path}")


In [ ]:
# ============================================================
# n2-13) Supplementary Annotation Quality & Marker Analysis
# ------------------------------------------------------------
# This step DOES NOT modify the existing annotation pipeline.
# It only adds additional analytical outputs that help:
#
# 1) Measure confidence of cell-type annotation
# 2) Identify cluster-specific marker genes
# 3) Evaluate consistency between clusters and annotations
#
# These outputs are valuable for downstream interpretation
# and are commonly requested in professional scRNA‑seq reports.
# ============================================================

import numpy as np
import pandas as pd
import scanpy as sc
from pathlib import Path

print("Running supplementary analysis (Step 13)...")

# -----------------------------
# Robust fallbacks for globals
# -----------------------------
# اگر نوت‌بوک از اول اجرا شده باشد این‌ها وجود دارند؛
# اگر سلول جداگانه اجرا شود، fallback می‌سازد که کرش نکند.
CONFIG = globals().get("CONFIG", {})
META = globals().get("META", {})
QC = globals().get("QC", {})
ANALYSIS = globals().get("ANALYSIS", {})
ANNOTATION = globals().get("ANNOTATION", {})
VIS = globals().get("VIS", {})

DATASET_NAME = globals().get("DATASET_NAME", META.get("project_name", "dataset"))
RESULTS_DIR = globals().get("RESULTS_DIR", Path("results") / DATASET_NAME)
FIG_DIR = globals().get("FIG_DIR", Path("figures") / DATASET_NAME)
CLUSTER_KEY = globals().get("CLUSTER_KEY", CONFIG.get("cluster_key", "leiden"))
RANK_GENES_METHOD = globals().get("RANK_GENES_METHOD", CONFIG.get("rank_genes_method", "wilcoxon"))

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

# =========================================================
# a) Annotation confidence score
# =========================================================

# فرض: برای هر سل‌تایپ ستون‌هایی مثل score_T_cell، score_B_cell در adata.obs داریم.
score_cols = [col for col in adata.obs.columns if col.startswith("score_")]

if len(score_cols) >= 2:
    score_matrix = adata.obs[score_cols].to_numpy()
    sorted_scores = np.sort(score_matrix, axis=1)
    confidence = sorted_scores[:, -1] - sorted_scores[:, -2]
    adata.obs["annotation_confidence"] = confidence
    print("annotation_confidence added to adata.obs")
else:
    print("Not enough score_* columns found; annotation_confidence will not be computed.")
    adata.obs["annotation_confidence"] = np.nan

# =========================================================
# b) Cluster marker genes (per cluster)
# =========================================================

markers_csv_path = RESULTS_DIR / f"8-{DATASET_NAME}_cluster_markers.csv"

if CLUSTER_KEY in adata.obs:
    print(f"Computing cluster marker genes using key = '{CLUSTER_KEY}' ...")
    
    if "rank_genes_groups" not in adata.uns:
        sc.tl.rank_genes_groups(
            adata,
            groupby=CLUSTER_KEY,
            method=RANK_GENES_METHOD,
            n_genes=50
        )
    
    markers_df = sc.get.rank_genes_groups_df(adata, group=None)
    markers_df.to_csv(markers_csv_path, index=False)
    print(f"Cluster marker genes saved to {markers_csv_path}")
else:
    print(f"Cluster key '{CLUSTER_KEY}' not found in adata.obs; skipping rank_genes_groups.")

# =========================================================
# c) Cluster vs Cell-Type crosstab
# =========================================================

ctab_path = RESULTS_DIR / f"9-{DATASET_NAME}_cluster_celltype_table.csv"
summary_path = RESULTS_DIR / f"10-{DATASET_NAME}_cluster_annotation_consistency.csv"

if CLUSTER_KEY in adata.obs and "cell_type" in adata.obs:
    print("Computing cluster vs cell-type crosstab...")
    ctab = pd.crosstab(adata.obs[CLUSTER_KEY], adata.obs["cell_type"])
    ctab.to_csv(ctab_path)
    print(f"Cluster vs cell-type table saved to {ctab_path}")

    # 13.4 Dominant cell type per cluster
    summary_rows = []
    for cluster, row in ctab.iterrows():
        total = row.sum()
        if total == 0:
            continue
        top_celltype = row.idxmax()
        top_count = row.max()
        top_fraction = top_count / total
        summary_rows.append(
            {
                "cluster": cluster,
                "dominant_cell_type": top_celltype,
                "n_cells": int(total),
                "n_cells_dominant": int(top_count),
                "fraction_dominant": float(top_fraction),
            }
        )
    summary_df = pd.DataFrame(summary_rows)
    summary_df.to_csv(summary_path, index=False)
    print(f"Cluster annotation consistency saved to {summary_path}")
else:
    print("Either cluster key or cell_type missing; skipping crosstab and consistency analysis.")

# =========================================================
# d) UMAP colored by annotation_confidence
# =========================================================

if "X_umap" in adata.obsm and "annotation_confidence" in adata.obs:
    import matplotlib.pyplot as plt

    umap_conf_path = FIG_DIR / f"11-{DATASET_NAME}_UMAP_annotation_confidence.png"
    sc.pl.umap(
        adata,
        color="annotation_confidence",
        cmap="viridis",
        show=False
    )
    plt.savefig(umap_conf_path, dpi=300, bbox_inches="tight")
    plt.close()
    print(f"UMAP colored by annotation confidence saved to {umap_conf_path}")
else:
    print("UMAP or annotation_confidence not available; skipping confidence UMAP.")

print("Supplementary analysis (Step 13) completed.\n")


In [ ]:
# =========================================================
# n2-14) Automated Analysis Report (PDF)
# =========================================================

from pathlib import Path
from datetime import datetime

import pandas as pd

# -----------------------------
# Robust fallbacks for globals
# -----------------------------
CONFIG = globals().get("CONFIG", {})
META = globals().get("META", {})
QC = globals().get("QC", {})
ANALYSIS = globals().get("ANALYSIS", {})
ANNOTATION = globals().get("ANNOTATION", {})
VIS = globals().get("VIS", {})

DATASET_NAME = globals().get("DATASET_NAME", META.get("project_name", "dataset"))
RESULTS_DIR = globals().get("RESULTS_DIR", Path("results") / DATASET_NAME)
FIG_DIR = globals().get("FIG_DIR", Path("figures") / DATASET_NAME)

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

# -----------------------------
# ReportLab imports
# -----------------------------
from reportlab.lib.pagesizes import A4
from reportlab.lib.units import cm
from reportlab.lib import colors
from reportlab.platypus import (
    SimpleDocTemplate,
    Paragraph,
    Spacer,
    Image,
    Table,
    TableStyle,
)
from reportlab.lib.styles import getSampleStyleSheet

print("Building automated PDF report (Step 14)...")

report_path = RESULTS_DIR / f"11-{DATASET_NAME}_scRNAseq_analysis_report_pro.pdf"
doc = SimpleDocTemplate(
    str(report_path),
    pagesize=A4,
    leftMargin=2 * cm,
    rightMargin=2 * cm,
    topMargin=2 * cm,
    bottomMargin=2 * cm,
)

styles = getSampleStyleSheet()
style_title = styles["Title"]
style_heading = styles["Heading2"]
style_body = styles["BodyText"]

story = []

# =========================================================
# 14.1 Title page
# =========================================================
title_text = f"scRNA-seq Analysis Report<br/>{DATASET_NAME}"
story.append(Paragraph(title_text, style_title))
story.append(Spacer(1, 0.7 * cm))

meta_lines = []
meta_lines.append(f"Project name: {META.get('project_name', DATASET_NAME)}")
meta_lines.append(f"Tissue / dataset type: {META.get('tissue_type', CONFIG.get('config_name', 'N/A'))}")
meta_lines.append(f"Species: {META.get('species', 'N/A')}")
meta_lines.append(f"Date: {datetime.now().strftime('%Y-%m-%d')}")
meta_lines.append(f"Results directory: {RESULTS_DIR}")
meta_lines.append(f"Figures directory: {FIG_DIR}")
meta_text = "<br/>".join(meta_lines)
story.append(Paragraph(meta_text, style_body))
story.append(Spacer(1, 1.0 * cm))

# =========================================================
# 14.2 Pipeline overview
# =========================================================
story.append(Paragraph("1. Pipeline Overview", style_heading))
overview = (
    "This report summarizes a standardized single-cell RNA-seq analysis pipeline, including: "
    "quality control, normalization and log-transformation, dimensionality reduction (PCA and UMAP), "
    "graph-based clustering (Leiden), marker-gene analysis, automatic cell-type annotation based on "
    "reference markers, and supplementary evaluation of annotation confidence and cluster consistency."
)
story.append(Paragraph(overview, style_body))
story.append(Spacer(1, 0.5 * cm))

# =========================================================
# 14.3 QC parameters + QC figures
# =========================================================
story.append(Paragraph("2. Quality Control (QC)", style_heading))

qc_lines = []
qc_lines.append("<b>QC parameters (from YAML)</b>")
for key in ["min_genes_per_cell", "max_genes_per_cell", "max_pct_mt", "min_cells_per_gene"]:
    if key in QC:
        qc_lines.append(f"{key}: {QC[key]}")
qc_text = "<br/>".join(qc_lines) if qc_lines else "QC parameters not available."
story.append(Paragraph(qc_text, style_body))
story.append(Spacer(1, 0.3 * cm))

# Try to embed QC figures if they exist
qc_figs = [
    FIG_DIR / f"1-{DATASET_NAME}_QC_violin.png",
    FIG_DIR / f"2-{DATASET_NAME}_QC_pct_counts_mt.png",
    FIG_DIR / f"3-{DATASET_NAME}_QC_n_genes_by_counts.png",
]
for fig_path in qc_figs:
    if fig_path.is_file():
        story.append(Image(str(fig_path), width=14 * cm, height=8 * cm))
        story.append(Spacer(1, 0.3 * cm))

# =========================================================
# 14.4 UMAPs & clustering
# =========================================================
story.append(Paragraph("3. UMAP Embeddings & Clustering", style_heading))

umap_text = (
    "UMAP embeddings provide a 2D representation of the high-dimensional transcriptomic space. "
    "Cells are colored by QC status, clusters, or cell types to illustrate global structure and "
    "major cell populations."
)
story.append(Paragraph(umap_text, style_body))
story.append(Spacer(1, 0.3 * cm))

umap_figs = [
    FIG_DIR / f"6-{DATASET_NAME}_umap_qc.png",
    FIG_DIR / f"7-{DATASET_NAME}_Leiden_clustering_UMAP.png",
    FIG_DIR / f"8-{DATASET_NAME}_UMAP_marker_genes.png",
    FIG_DIR / f"11-{DATASET_NAME}_UMAP_annotation_confidence.png",
]
for fig_path in umap_figs:
    if fig_path.is_file():
        story.append(Image(str(fig_path), width=14 * cm, height=8 * cm))
        story.append(Spacer(1, 0.3 * cm))

# =========================================================
# 14.5 Cell-type annotation & marker genes
# =========================================================
story.append(Paragraph("4. Cell-Type Annotation & Marker Genes", style_heading))

ref_markers = ANNOTATION.get("reference_markers", {})
if ref_markers:
    n_ct = len(ref_markers)
    story.append(
        Paragraph(
            f"Automatic cell-type annotation was performed using {n_ct} reference marker sets "
            "defined in the YAML configuration. Each cell was assigned the label whose marker "
            "gene score was highest.",
            style_body,
        )
    )
else:
    story.append(
        Paragraph(
            "No reference markers were defined in the YAML configuration; annotation may be performed manually.",
            style_body,
        )
    )
story.append(Spacer(1, 0.3 * cm))

# Top markers table from cluster marker CSV (Step 13)
markers_csv_path = RESULTS_DIR / f"8-{DATASET_NAME}_cluster_markers.csv"
if markers_csv_path.is_file():
    try:
        markers_df = pd.read_csv(markers_csv_path)
        # take top 3 markers per cluster for a compact table
        top_markers = markers_df.groupby("group").head(3)

        table_data = [["cluster", "gene", "logFC", "pval_adj"]]
        for _, row in top_markers.iterrows():
            table_data.append(
                [
                    str(row.get("group", "")),
                    str(row.get("names", "")),
                    f"{row.get('logfoldchanges', 0):.2f}",
                    f"{row.get('pvals_adj', 0):.1e}",
                ]
            )

        tbl = Table(table_data, hAlign="LEFT")
        tbl.setStyle(
            TableStyle(
                [
                    ("BACKGROUND", (0, 0), (-1, 0), colors.lightgrey),
                    ("GRID", (0, 0), (-1, -1), 0.25, colors.grey),
                    ("FONTNAME", (0, 0), (-1, 0), "Helvetica-Bold"),
                ]
            )
        )
        story.append(tbl)
        story.append(Spacer(1, 0.3 * cm))
    except Exception as e:
        story.append(Paragraph(f"Could not load marker table: {e}", style_body))

# =========================================================
# 14.6 Cluster vs cell-type consistency
# =========================================================
summary_path = RESULTS_DIR / f"10-{DATASET_NAME}_cluster_annotation_consistency.csv"
if summary_path.is_file():
    try:
        summary_df = pd.read_csv(summary_path)
        story.append(Paragraph("5. Cluster vs Cell-Type Consistency", style_heading))

        table_data = [["cluster", "dominant cell type", "n_cells", "fraction_dominant"]]
        for _, row in summary_df.iterrows():
            table_data.append(
                [
                    str(row["cluster"]),
                    str(row["dominant_cell_type"]),
                    int(row["n_cells"]),
                    f"{row['fraction_dominant']:.2f}",
                ]
            )
        tbl = Table(table_data, hAlign="LEFT")
        tbl.setStyle(
            TableStyle(
                [
                    ("BACKGROUND", (0, 0), (-1, 0), colors.lightgrey),
                    ("GRID", (0, 0), (-1, -1), 0.25, colors.grey),
                    ("FONTNAME", (0, 0), (-1, 0), "Helvetica-Bold"),
                ]
            )
        )
        story.append(tbl)
        story.append(Spacer(1, 0.3 * cm))
    except Exception as e:
        story.append(Paragraph(f"Could not load cluster-consistency table: {e}", style_body))

# =========================================================
# 14.7 Appendix — key parameters from YAML
# =========================================================
story.append(Paragraph("6. Key Analysis Parameters (from YAML)", style_heading))

param_lines = []

param_lines.append("<b>QC</b>")
for k, v in QC.items():
    param_lines.append(f"{k}: {v}")

param_lines.append("<br/><b>Analysis</b>")
for k, v in ANALYSIS.items():
    param_lines.append(f"{k}: {v}")

param_lines.append("<br/><b>Annotation</b>")
param_lines.append(f"reference marker groups: {len(ref_markers)}")

param_text = "<br/>".join(param_lines) if param_lines else "No parameters available."
story.append(Paragraph(param_text, style_body))

story.append(Paragraph("Total cells: {}".format(adata.n_obs), style_body))
story.append(Paragraph("Total genes: {}".format(adata.n_vars), style_body))

# =========================================================
# Build document
# =========================================================
doc.build(story)
print(f"PDF report saved to: {report_path}\n")


In [ ]:
## END of notebook 2- Annotation

In [ ]:
# --------------------------------------------

In [ ]:
## notebook 3 - GO Enrichment Analysis (BP, MF, CC) 

In [ ]:
# =============================================================================
# n3-15) Enrichr connectivity check (refined)
# =============================================================================

import time
import requests
import gseapy as gp

# -----------------------------------------------------------------------------
# Settings
# -----------------------------------------------------------------------------

GO_GENE_SETS = [
    "GO_Biological_Process_2021",
    "GO_Molecular_Function_2021",
    "GO_Cellular_Component_2021",
]

PVAL_THRESH_DEFAULT = 0.05
TOP_N_DEFAULT = 100

ORGANISM_DEFAULT = adata.uns.get("organism", "human")

ENRICHR_CHECK_TIMEOUT = 10
ENRICHR_SINGLE_TIMEOUT = 5
ENRICHR_TEST_URL = "https://maayanlab.cloud/Enrichr/datasetStatistics"

ENRICHR_AVAILABLE = False

print("[INFO] GO Enrichment environment initialized.")
print(f"[INFO] GO gene sets: {GO_GENE_SETS}")
print(f"[INFO] Organism: {ORGANISM_DEFAULT}")
print(f"[INFO] Checking Enrichr connectivity (timeout={ENRICHR_CHECK_TIMEOUT}s)...")

# -----------------------------------------------------------------------------
# Connectivity test
# -----------------------------------------------------------------------------
def _try_enrichr_connection(timeout=ENRICHR_SINGLE_TIMEOUT):
    try:
        resp = requests.get(ENRICHR_TEST_URL, timeout=timeout)
        return resp.status_code == 200
    except Exception:
        return False

# -----------------------------------------------------------------------------
# Retry loop
# -----------------------------------------------------------------------------
start_time = time.time()

while (time.time() - start_time) < ENRICHR_CHECK_TIMEOUT:
    if _try_enrichr_connection():
        ENRICHR_AVAILABLE = True
        break
    time.sleep(0.5)

# -----------------------------------------------------------------------------
# Final status
# -----------------------------------------------------------------------------
if ENRICHR_AVAILABLE:
    print("[INFO] ✅ Enrichr is reachable. GO enrichment will run.")
else:
    print("[WARNING] ❌ Enrichr not reachable.")
    print("[WARNING] GO enrichment will be skipped.")

In [ ]:
# =============================================================================
# n3-16) Helper function: Run GO enrichment (BP, MF, CC) for a given group (DEBUGGED)
# =============================================================================

def run_go_enrichment_for_group(
    gene_table,
    group_name,
    gene_col="names",
    pval_col="pvals_adj",
    pval_thresh=PVAL_THRESH_DEFAULT,
    top_n=TOP_N_DEFAULT,
    organism=ORGANISM_DEFAULT,
    gene_sets=GO_GENE_SETS,
    verbose=True
):

    if verbose:
        print("\n" + "="*60)
        print(f"[GO - {group_name}] START")
        print("="*60)

    # -------------------------------------------------------------------------
    # 16-1. Input validation
    # -------------------------------------------------------------------------
    if gene_table is None or gene_table.empty:
        print(f"[GO - {group_name}] ❌ Input gene table is empty.")
        return None

    if verbose:
        print(f"[GO - {group_name}] Input genes: {gene_table.shape[0]} rows")

    if gene_col not in gene_table.columns or pval_col not in gene_table.columns:
        raise ValueError(
            f"[GO - {group_name}] Columns '{gene_col}' or '{pval_col}' not found. "
            f"Available: {list(gene_table.columns)}"
        )

    # -------------------------------------------------------------------------
    # 16-2. Filter genes
    # -------------------------------------------------------------------------
    df = gene_table.sort_values(pval_col, ascending=True)

    # check logFC existence
    if "logfoldchanges" in df.columns:
        df = df.query(f"{pval_col} <= @pval_thresh and logfoldchanges > 0")
        if verbose:
            print(f"[GO - {group_name}] Filtering with logFC > 0")
    else:
        df = df.query(f"{pval_col} <= @pval_thresh")
        if verbose:
            print(f"[GO - {group_name}] ⚠️ 'logfoldchanges' not found → ignoring FC filter")

    if verbose:
        print(f"[GO - {group_name}] Genes after filtering: {df.shape[0]}")

    df = df.head(top_n)

    genes = df[gene_col].dropna().astype(str).unique().tolist()

    if verbose:
        print(f"[GO - {group_name}] Unique genes used: {len(genes)}")

    if len(genes) < 10:
        print(f"[GO - {group_name}] ❌ Too few genes ({len(genes)}). Skipping.")
        return None

    # -------------------------------------------------------------------------
    # 16-3. Connectivity check
    # -------------------------------------------------------------------------
    if not ENRICHR_AVAILABLE:
        print(f"[GO - {group_name}] ⚠️ Enrichr unavailable → skipping.")
        return None

    # -------------------------------------------------------------------------
    # 16-4. Run enrichment
    # -------------------------------------------------------------------------
    print(f"[GO - {group_name}] Running Enrichr...")

    try:
        enr = gp.enrichr(
            gene_list=genes,
            gene_sets=gene_sets,
            organism=organism,
            outdir=None,
            cutoff=0.05
        )
    except Exception as e:
        print(f"[GO - {group_name}] ❌ Enrichr error: {e}")
        return None

    # -------------------------------------------------------------------------
    # 16-5. Validate output
    # -------------------------------------------------------------------------
    if enr is None or not hasattr(enr, "results") or enr.results is None:
        print(f"[GO - {group_name}] ❌ No result object returned.")
        return None

    if enr.results.empty:
        print(f"[GO - {group_name}] ⚠️ No enriched terms found.")
        return None

    # -------------------------------------------------------------------------
    # 16-6. Final formatting
    # -------------------------------------------------------------------------
    res = enr.results.copy()

    res["group"] = group_name
    res["n_genes"] = len(genes)
    res["organism"] = organism

    if "Gene_set" in res.columns:
        res["go_library"] = res["Gene_set"]
    elif "gene_set" in res.columns:
        res["go_library"] = res["gene_set"]
    else:
        res["go_library"] = "GO_mixed"

    res["input_genes"] = ",".join(genes[:20])

    print(f"[GO - {group_name}] ✅ SUCCESS: {res.shape[0]} terms found")

    return res

print("[INFO] GO enrichment function is defined and ready to use.")

In [ ]:
'''
test_df = markers_df.head(200)

run_go_enrichment_for_group(
    gene_table=test_df,
    group_name="TEST"
)
'''

In [ ]:
# =============================================================================
# n3-17) Run GO Enrichment (DEBUGGED VERSION)
# =============================================================================

print("\n[STEP 17] Starting GO enrichment pipeline...")

# -----------------------------------------------------------------------------
# 17-1. Detect DE structure
# -----------------------------------------------------------------------------
try:
    markers_df
except NameError:
    print("[ERROR] markers_df is not defined.")
    groups_to_analyze = []
else:
    print(f"[INFO] markers_df shape: {markers_df.shape}")

    if markers_df.empty:
        print("[ERROR] markers_df is EMPTY.")
        groups_to_analyze = []

    elif {"group", "names", "pvals_adj"}.issubset(markers_df.columns):
        group_col = "group"
        gene_col = "names"
        pval_col = "pvals_adj"
        print("[DE] Using cluster markers")

    elif {"celltype", "names", "pvals_adj"}.issubset(markers_df.columns):
        group_col = "celltype"
        gene_col = "names"
        pval_col = "pvals_adj"
        print("[DE] Using cell-type markers")

    else:
        print("[ERROR] Could not detect DE columns.")
        print(f"Available columns: {list(markers_df.columns)}")
        groups_to_analyze = []

    if "group_col" in locals():
        groups_to_analyze = sorted(markers_df[group_col].dropna().unique())
        print(f"[INFO] Groups detected: {len(groups_to_analyze)}")
        print(groups_to_analyze)

# -----------------------------------------------------------------------------
# 17-2. Run enrichment
# -----------------------------------------------------------------------------
all_go_results = []

if not groups_to_analyze:
    print("[WARNING] No groups found → GO enrichment will NOT run.")

else:
    print("\n[INFO] Starting per-group GO analysis...\n")

    for g_name in groups_to_analyze:

        print("\n" + "-" * 70)
        print(f"[RUN] Group: {g_name}")
        print("-" * 70)

        subset_df = markers_df[markers_df[group_col] == g_name].copy()

        print(f"[INFO] Genes in this group: {subset_df.shape[0]}")

        go_res = run_go_enrichment_for_group(
            gene_table=subset_df,
            group_name=str(g_name),
            gene_col=gene_col,
            pval_col=pval_col,
            organism=adata.uns.get("organism", ORGANISM_DEFAULT),
            gene_sets=GO_GENE_SETS,
            verbose=True
        )

        if go_res is not None and not go_res.empty:
            all_go_results.append(go_res)

            out_individual = RESULTS_DIR / f"12-{DATASET_NAME}_GO_group_{g_name}.csv"
            go_res.to_csv(out_individual, index=False)

            print(f"[SAVE] {out_individual}")

        else:
            print(f"[GO - {g_name}] ❌ No enrichment result.")

# -----------------------------------------------------------------------------
# 17-3. Combine results
# -----------------------------------------------------------------------------
print("\n[INFO] Combining GO results...")

if all_go_results:

    go_all_df = pd.concat(all_go_results, axis=0, ignore_index=True)

    print(f"[INFO] Total enriched rows: {go_all_df.shape[0]}")

    if "Adjusted P-value" in go_all_df.columns:
        go_all_df = go_all_df.sort_values(["group", "Adjusted P-value"])

    try:
        from statsmodels.stats.multitest import multipletests
        if "Adjusted P-value" in go_all_df.columns:
            go_all_df["p_adj_global"] = multipletests(
                go_all_df["Adjusted P-value"],
                method="fdr_bh"
            )[1]
            print("[INFO] Global FDR added.")
    except ImportError:
        print("[WARNING] statsmodels not installed.")

    out_csv = RESULTS_DIR / f"13-{DATASET_NAME}_GO_enrichment_all_groups_BP_MF_CC.csv"
    go_all_df.to_csv(out_csv, index=False)

    print("\n" + "=" * 60)
    print(f"[OUTPUT] Saved: {out_csv}")
    print("=" * 60)

    print("\n[SUMMARY]")
    print(go_all_df.groupby("group").size())

else:
    print("[WARNING] No GO results were generated.")

In [ ]:
# =============================================================================
# n3-18) Advanced GO Visualization (Final polished version with legend)
# =============================================================================

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.lines import Line2D

print("[STEP 18] GO visualization started...")

# -----------------------------------------------------------------------------
# 18-0. Check input
# -----------------------------------------------------------------------------
try:
    go_all_df
except NameError:
    print("[ERROR] 'go_all_df' not found. Run GO enrichment step first.")
else:

    if go_all_df.empty:
        print("[INFO] GO results table is empty. Skipping visualization.")
    else:

        if "Adjusted P-value" not in go_all_df.columns:
            print("[ERROR] 'Adjusted P-value' column not found. Cannot plot.")
        else:

            print(f"[INFO] GO table size: {go_all_df.shape}")

            # -----------------------------------------------------------------------------
            # 18-1. Preprocessing
            # -----------------------------------------------------------------------------
            go_all_df["log10_padj"] = -np.log10(go_all_df["Adjusted P-value"] + 1e-300)

            def extract_go_category(x):
                if "Biological_Process" in str(x):
                    return "BP"
                elif "Molecular_Function" in str(x):
                    return "MF"
                elif "Cellular_Component" in str(x):
                    return "CC"
                return "Other"

            go_all_df["GO_cat"] = go_all_df["go_library"].apply(extract_go_category)

            def extract_gene_count(x):
                try:
                    return int(str(x).split("/")[0])
                except:
                    return 1

            if "Overlap" in go_all_df.columns:
                go_all_df["gene_count"] = go_all_df["Overlap"].apply(extract_gene_count)
            else:
                go_all_df["gene_count"] = 1

            # کوتاه کردن اسم GO term
            def shorten(term, max_len=60):
                term = str(term)
                return term if len(term) <= max_len else term[:max_len-3] + "..."

            # -----------------------------------------------------------------------------
            # Colors + legend
            # -----------------------------------------------------------------------------
            color_map = {
                "BP": "tab:blue",
                "MF": "tab:orange",
                "CC": "tab:green",
                "Other": "gray"
            }

            legend_elements = [
                Line2D([0], [0], marker='o', color='w',
                       label=cat,
                       markerfacecolor=color_map[cat],
                       markersize=8)
                for cat in ["BP", "MF", "CC"]
            ]

            groups = sorted(go_all_df["group"].unique())
            print(f"[INFO] Number of groups: {len(groups)}")

            fig_counter = 12

            # -----------------------------------------------------------------------------
            # 18-2. Per-group bubble plots
            # -----------------------------------------------------------------------------
            for g in groups:

                df_g = go_all_df[go_all_df["group"] == g].copy()

                if df_g.empty:
                    print(f"[WARN] Group {g} empty → skipped")
                    continue

                df_g = df_g.sort_values("Adjusted P-value").head(10)
                df_g = df_g.iloc[::-1]
                df_g["Term_short"] = df_g["Term"].apply(shorten)

                print(f"[PLOT] Group {g} → {len(df_g)} terms")

                plt.figure(figsize=(7, 4))

                plt.scatter(
                    df_g["log10_padj"],
                    range(len(df_g)),
                    s=df_g["gene_count"] * 25,
                    c=df_g["GO_cat"].map(color_map),
                    alpha=0.85
                )

                plt.yticks(range(len(df_g)), df_g["Term_short"])
                plt.xlabel("-log10(adj p-value)")
                plt.title(f"GO Bubble Plot (Group {g})")

                plt.legend(handles=legend_elements, title="GO Category")

                bubble_path = FIG_DIR / f"{fig_counter}-{DATASET_NAME}_GO_bubble_group_{g}.png"
                plt.savefig(bubble_path, dpi=300, bbox_inches="tight")
                plt.close()

                print(f"[FIG] Saved → {bubble_path}")
                fig_counter += 1

            # -----------------------------------------------------------------------------
            # 18-3. Combined plot
            # -----------------------------------------------------------------------------
            print("[PLOT] Combined GO plot")

            df_top = (
                go_all_df
                .sort_values("Adjusted P-value")
                .groupby("group")
                .head(5)
            )

            df_top["Term_short"] = df_top["Term"].apply(shorten)

            plt.figure(figsize=(8, 5))

            plt.scatter(
                df_top["group"],
                df_top["Term_short"],
                s=df_top["gene_count"] * 25,
                c=df_top["GO_cat"].map(color_map),
                alpha=0.8
            )

            plt.xlabel("Group")
            plt.ylabel("GO Term")
            plt.title("GO Enrichment Across Groups")

            plt.legend(handles=legend_elements, title="GO Category")

            combined_path = FIG_DIR / f"{fig_counter}-{DATASET_NAME}_GO_combined_bubble.png"
            plt.savefig(combined_path, dpi=300, bbox_inches="tight")
            plt.close()

            print(f"[FIG] Saved → {combined_path}")
            fig_counter += 1

            # -----------------------------------------------------------------------------
            # 18-4. Category-specific barplots
            # -----------------------------------------------------------------------------
            for cat in ["BP", "MF", "CC"]:

                df_cat = go_all_df[go_all_df["GO_cat"] == cat]

                if df_cat.empty:
                    print(f"[INFO] No terms for {cat}")
                    continue

                df_cat = df_cat.sort_values("Adjusted P-value").head(15)
                df_cat["Term_short"] = df_cat["Term"].apply(shorten)

                print(f"[PLOT] {cat} barplot → {len(df_cat)} terms")

                plt.figure(figsize=(7, 5))

                plt.barh(
                    df_cat["Term_short"],
                    df_cat["log10_padj"],
                )

                plt.xlabel("-log10(adj p-value)")
                plt.title(f"Top {cat} Terms")

                cat_path = FIG_DIR / f"{fig_counter}-{DATASET_NAME}_GO_{cat}_barplot.png"
                plt.savefig(cat_path, dpi=300, bbox_inches="tight")
                plt.close()

                print(f"[FIG] Saved → {cat_path}")
                fig_counter += 1

            print("\n[STEP 18 COMPLETED] GO visualization finished successfully.")

In [ ]:
##  End of notebook 3 - GO Enrichment Analysis (BP, MF, CC) 

In [ ]:
## ---------------------------------------------